# NB_hospital_daily_summary
Builds `caresync_databricks.gold.hospital_daily_summary` per STTM sheet **hospital_daily_summary**.
* **Grain:** one row per hospital per calendar day
* **Driving table:** `silver.lab_results` grouped by `hospital_id`, `test_date`
* **Joins:** LEFT JOIN `silver.hospitals`; LEFT JOIN calendar on date key
* **Window:** LAG(total_tests) by hospital_id ordered by summary_date (day-over-day trend)
* **Load:** FULL, CREATE OR REPLACE TABLE ... AS SELECT
* **Data note:** the STTM names the calendar `silver.calendar` with key `date_key`. The actual table is `silver.calender` with INT key `datekey`, so that is what this notebook uses.

In [0]:
%sql
CREATE OR REPLACE TABLE caresync_databricks.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab test volume, abnormal/critical rates and day-over-day trend (STTM hospital_daily_summary)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM caresync_databricks.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                        AS hospital_id,
  h.hospital_name,
  w.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(w.total_tests AS INT)                                        AS total_tests,
  CAST(w.unique_patients_tested AS INT)                             AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                AS critical_test_count,
  CAST(w.abnormal_test_count / w.total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(w.critical_test_count / w.total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                               AS prev_day_total_tests,
  CAST(
    CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
         ELSE (w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100
    END AS DECIMAL(6,2))                                            AS test_volume_change_pct
FROM with_lag w
LEFT JOIN caresync_databricks.silver.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN caresync_databricks.silver.calender c   -- STTM: silver.calendar.date_key
  ON c.datekey = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT);

In [0]:
NB_hospital_daily_summary_df = spark.table("caresync_databricks.gold.hospital_daily_summary")
record_count_hospital_daily_summary = NB_hospital_daily_summary_df.count()
dbutils.notebook.exit(str(record_count_hospital_daily_summary))